## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [ ]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [3]:
load_dotenv(override=True)
openai = OpenAI()

In [4]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [ ]:
print(linkedin)

In [5]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [6]:
name = "Ed Donner"

In [7]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [ ]:
system_prompt

In [9]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [ ]:
gr.ChatInterface(chat, type="messages").launch()

## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

In [11]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [23]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [24]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [25]:
import os
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [26]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.0-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [27]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content

In [ ]:
reply

In [ ]:
evaluate(reply, "do you hold a patent?", messages[:1])

In [30]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [35]:
def chat(message, history):
    if "patent" in message:
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

In [ ]:
gr.ChatInterface(chat, type="messages").launch()

# 📊 Algorithm Workflow & Concept Explanation

## 🎯 Main Algorithm: **Self-Evaluating & Self-Correcting Chatbot**

This lab implements a **quality-controlled chatbot** that evaluates its own responses and retries if the quality is not acceptable.

---

## 🔄 Complete Workflow Diagram

```
┌─────────────────────────────────────────────────────────────────┐
│                    USER SENDS A MESSAGE                         │
└────────────────────────────┬────────────────────────────────────┘
                             │
                             ▼
┌─────────────────────────────────────────────────────────────────┐
│  STEP 1: GENERATE INITIAL REPLY                                 │
│  ────────────────────────────────────────────────────────────   │
│  • Load context: LinkedIn PDF + Summary                          │
│  • Create system prompt with persona instructions               │
│  • Send to LLM (GPT-4o-mini) to generate reply                  │
└────────────────────────────┬────────────────────────────────────┘
                             │
                             ▼
┌─────────────────────────────────────────────────────────────────┐
│  STEP 2: EVALUATE THE REPLY                                     │
│  ────────────────────────────────────────────────────────────   │
│  • Send reply + conversation history to Evaluator LLM            │
│  • Evaluator checks:                                            │
│    ✓ Is response professional?                                  │
│    ✓ Does it match the persona?                                │
│    ✓ Is it accurate based on context?                          │
│  • Returns: {is_acceptable: bool, feedback: str}              │
└────────────────────────────┬────────────────────────────────────┘
                             │
                    ┌────────┴────────┐
                    │                 │
            is_acceptable?      NOT acceptable
                    │                 │
                    ▼                 ▼
        ┌──────────────────┐  ┌──────────────────┐
        │  STEP 3A:        │  │  STEP 3B:        │
        │  RETURN REPLY    │  │  RETRY WITH      │
        │  (Success!)      │  │  FEEDBACK        │
        └──────────────────┘  └────────┬─────────┘
                                       │
                                       ▼
                          ┌──────────────────────────┐
                          │  • Add feedback to       │
                          │    system prompt         │
                          │  • Include rejected      │
                          │    answer                │
                          │  • Regenerate reply      │
                          │  • Return new reply      │
                          └──────────────────────────┘
```

---

## 🧠 Key Concepts Explained

### 1. **Dual LLM Architecture**
   - **Agent LLM**: Generates responses (GPT-4o-mini)
   - **Evaluator LLM**: Judges quality (Gemini 2.0 Flash)
   - Why separate? Different models can have different strengths!

### 2. **Self-Evaluation Loop**
   ```
   Generate → Evaluate → Accept/Retry → Return
   ```
   - The system checks its own work before showing to user
   - Similar to human: "Did I answer correctly? Let me check..."

### 3. **Feedback-Informed Retry**
   - When evaluation fails, the feedback is added to the system prompt
   - The LLM learns from its mistake: "You said X, but that was wrong because Y"
   - This is like a **self-correction mechanism**

### 4. **Context Management**
   - **Persona Context**: LinkedIn profile + summary
   - **Conversation History**: Previous messages
   - **Evaluation Context**: What went wrong (if retrying)

---

## 💡 Real-World Analogy

Think of it like a **quality control system in a factory**:

1. **Worker (Agent LLM)**: Produces a product (generates reply)
2. **Inspector (Evaluator LLM)**: Checks quality (evaluates reply)
3. **Feedback Loop**: If product fails inspection, worker gets feedback and tries again
4. **Final Product**: Only acceptable products reach the customer

---

## 🔑 Why This Matters

This pattern is the foundation of **agentic AI systems**:
- **Self-monitoring**: Agents can check their own work
- **Self-improvement**: Agents learn from feedback
- **Quality assurance**: Better outputs without human intervention
- **Reliability**: Reduces errors before they reach users

---

## 📝 Code Flow Summary

```python
def chat(message, history):
    # 1. Generate initial reply
    reply = generate_reply(message, history)
    
    # 2. Evaluate the reply
    evaluation = evaluate(reply, message, history)
    
    # 3. Decision point
    if evaluation.is_acceptable:
        return reply  # ✅ Success!
    else:
        # 4. Retry with feedback
        reply = rerun(reply, message, history, evaluation.feedback)
        return reply  # 🔄 Improved version
```

---

## 🎓 Learning Points

1. **Separation of Concerns**: Generation vs. Evaluation
2. **Iterative Improvement**: Using feedback to improve
3. **Structured Output**: Using Pydantic for evaluation results
4. **Error Handling**: Gracefully handling failures
5. **Context Engineering**: How to structure prompts for best results


# 🔍 Parameter Flow: Where `reply`, `message`, and `history` Come From

## 📥 Parameter Origins & Flow

### 1. **`message`** - The User's Current Input

**Source**: Automatically provided by **Gradio's ChatInterface**

```python
gr.ChatInterface(chat, type="messages").launch()
#                    ↑
#         Gradio calls chat(message, history)
#         when user types and sends a message
```

**What it is**: 
- A **string** containing the user's current message/question
- Example: `"do you hold a patent?"`

**Why we need it**: 
- This is what the user just typed
- We need it to generate a response
- We need it for evaluation context

---

### 2. **`history`** - Previous Conversation

**Source**: Automatically provided by **Gradio's ChatInterface**

**What it is**: 
- A **list of message dictionaries** containing the entire conversation history
- Format: `[{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}, ...]`
- Example:
  ```python
  [
      {"role": "user", "content": "What's your background?"},
      {"role": "assistant", "content": "I'm a software engineer..."},
      {"role": "user", "content": "Do you have a patent?"}  # This becomes 'message'
  ]
  ```

**Why we need it**: 
- LLMs need context to have coherent conversations
- Without history, each response would be isolated
- The evaluator needs history to judge if the reply makes sense in context

**Important Note**: 
- On the **first message**, `history` is an **empty list** `[]`
- Gradio automatically builds this list as the conversation progresses

---

### 3. **`reply`** - The Generated Response

**Source**: **Generated INSIDE the `chat()` function** (NOT from Gradio)

**What it is**: 
- A **string** containing the LLM's generated response
- Created by calling the OpenAI API

**Where it's created**:
```python
def chat(message, history):
    # ... build messages ...
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply = response.choices[0].message.content  # ← reply is created HERE
    #        ↑
    #    This is the generated text from the LLM
```

**Why we need it**: 
- This is what we want to evaluate
- This is what we might need to retry if evaluation fails
- This is what gets returned to the user (via Gradio)

---

## 🔄 Complete Parameter Flow Diagram

```
┌─────────────────────────────────────────────────────────────┐
│  USER TYPES IN GRADIO UI: "Do you hold a patent?"           │
└────────────────────────────┬────────────────────────────────┘
                             │
                             ▼
┌─────────────────────────────────────────────────────────────┐
│  GRADIO ChatInterface AUTOMATICALLY CALLS:                  │
│                                                              │
│  chat(message="Do you hold a patent?",                      │
│       history=[previous conversation messages])              │
│                                                              │
│  ════════════════════════════════════════════════════════   │
│  message ← From Gradio (user input)                        │
│  history ← From Gradio (conversation so far)                │
└────────────────────────────┬────────────────────────────────┘
                             │
                             ▼
┌─────────────────────────────────────────────────────────────┐
│  INSIDE chat() FUNCTION:                                    │
│                                                              │
│  1. Build messages list:                                    │
│     messages = [system_prompt] + history + [message]        │
│                                                              │
│  2. Call LLM API:                                           │
│     response = openai.chat.completions.create(...)          │
│                                                              │
│  3. Extract reply:                                          │
│     reply = response.choices[0].message.content             │
│     ↑                                                        │
│     reply ← Generated HERE (not from Gradio!)               │
└────────────────────────────┬────────────────────────────────┘
                             │
                             ▼
┌─────────────────────────────────────────────────────────────┐
│  CALL evaluate() FUNCTION:                                  │
│                                                              │
│  evaluate(reply, message, history)                          │
│    ↑        ↑       ↑                                        │
│    │        │       └─ From chat() parameters               │
│    │        └───────── From chat() parameters               │
│    └────────────────── Generated inside chat()              │
└────────────────────────────┬────────────────────────────────┘
                             │
                             ▼
┌─────────────────────────────────────────────────────────────┐
│  IF EVALUATION FAILS, CALL rerun():                         │
│                                                              │
│  rerun(reply, message, history, feedback)                   │
│    ↑        ↑       ↑        ↑                              │
│    │        │       │        └─ From evaluation result      │
│    │        │       └────────── From chat() parameters      │
│    │        └────────────────── From chat() parameters      │
│    └─────────────────────────── Generated inside chat()     │
└────────────────────────────┬────────────────────────────────┘
                             │
                             ▼
┌─────────────────────────────────────────────────────────────┐
│  RETURN reply TO GRADIO                                     │
│  Gradio displays it in the chat interface                   │
└─────────────────────────────────────────────────────────────┘
```

---

## 📋 Parameter Summary Table

| Parameter | Source | Type | Purpose | Example |
|-----------|--------|------|---------|---------|
| **`message`** | Gradio ChatInterface | `str` | Current user input | `"Do you hold a patent?"` |
| **`history`** | Gradio ChatInterface | `list[dict]` | Previous conversation | `[{"role": "user", "content": "..."}, ...]` |
| **`reply`** | Generated in `chat()` | `str` | LLM's response | `"Yes, I hold a patent for..."` |

---

## 🎯 Key Insights

1. **Gradio is the Entry Point**: 
   - Gradio automatically provides `message` and `history` when user interacts
   - You don't manually pass these - Gradio handles it!

2. **`reply` is Internal**: 
   - `reply` is created inside your code, not from outside
   - It's the output of the LLM that you then evaluate

3. **Parameter Passing Chain**:
   ```
   Gradio → chat(message, history) 
         → generates reply 
         → evaluate(reply, message, history)
         → rerun(reply, message, history, feedback)
   ```

4. **Why This Design?**
   - **Separation of concerns**: Gradio handles UI, your code handles logic
   - **Reusability**: Functions can be called independently
   - **Testability**: You can test functions without Gradio UI

---

## 💻 Code Example: Tracing the Flow

```python
# Step 1: User types "Hello" in Gradio UI
# Gradio automatically calls:
chat(message="Hello", history=[])

# Step 2: Inside chat(), reply is generated
def chat(message, history):
    # message = "Hello" (from Gradio)
    # history = [] (from Gradio, empty on first message)
    
    messages = [system_prompt] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(...)
    reply = response.choices[0].message.content  # reply = "Hi! How can I help?"
    
    # Step 3: Pass all three to evaluate()
    evaluation = evaluate(reply, message, history)
    #         ↑      ↑       ↑
    #         │      │       └─ from chat() parameter
    #         │      └───────── from chat() parameter  
    #         └──────────────── generated above
    
    return reply  # Return to Gradio for display
```

---

## 🔗 Connection to Gradio

**Gradio's ChatInterface signature**:
```python
ChatInterface(fn, ...)
#            ↑
#    fn must be: fn(message: str, history: list) -> str
```

**Your function matches this**:
```python
def chat(message, history):  # ← Matches Gradio's expected signature!
    # ... your code ...
    return reply  # ← Returns string for Gradio to display
```

This is why Gradio can automatically call your function with the right parameters!
